# 07 — Multi-Document RAG

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Combine narrative documents (PDFs) with structured documents (Excel/CSV) in a single store.
2. Tag each chunk with `doc_type` metadata.
3. Use **metadata filters** to search within a category ("only policies").
4. Compare facts across documents (cross-document Q&A).


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 7.1 — Load PDFs + spreadsheets into one store

In [ ]:
from pathlib import Path
from src.document_loaders import load_pdfs_in_folder, excel_to_text_documents, csv_to_text_documents
from src.rag_utils import VectorStore, chunk_documents

docs = []
docs.extend(load_pdfs_in_folder('data/generated/pdf'))
for p in sorted(Path('data/generated/xlsx').glob('*.xlsx')):
    docs.extend(excel_to_text_documents(p))
for p in sorted(Path('data/generated/csv').glob('*.csv')):
    docs.extend(csv_to_text_documents(p))
print(f'Loaded {len(docs)} documents (PDF pages + sheets + CSV summaries)')

chunks = chunk_documents(docs, chunk_size=800, overlap=120)
store = VectorStore()
store.add(chunks)
print(f'Store has {len(store)} chunks.')

## 7.2 — Inspect the doc_type distribution

In [ ]:
from collections import Counter
print(Counter(c['metadata'].get('doc_type','?') for c in chunks))

## 7.3 — Filtered retrieval — only policies

In [ ]:
from src.rag_utils import rag_answer
answer = rag_answer(
    'What is the approval threshold for related-party purchases?',
    store, k=4, where={'doc_type': 'policy'},
)
print(answer)

## 7.4 — Cross-document question

A question that requires combining the loan agreement *and* the annual report.

In [ ]:
answer, hits = rag_answer(
    'Compare the loan terms in the loan agreement summary with what the annual report discloses about borrowings.',
    store, k=6, return_hits=True,
)
print(answer)
print('\nSources consulted:')
for h in hits:
    print('  -', h['metadata'].get('source'), 'p.', h['metadata'].get('page'))

## 7.5 — Find inconsistencies

*"Is the related-party listing consistent with what the board minutes approved?"*

In [ ]:
print(rag_answer(
    'Are all related-party transactions in the RPT listing supported by board approval recorded in the minutes? Flag any without approval.',
    store, k=6,
))

## 7.6 — Ledger-supported audit risks

In [ ]:
print(rag_answer(
    'Which audit risks mentioned in the planning memo are supported by patterns visible in the ledger data?',
    store, k=6,
))

## Expected output

* 7.3 — answer cites the Procurement Policy and Internal Control Policy.
* 7.4 — answer cites both `06_loan_agreement_summary.pdf` *and* `01_annual_report_extract.pdf`.
* 7.5 — should flag the row dated 2082-02-20 (Himal Family Enterprises) as not Board-approved.


## Exercise

1. Build a `doc_type` filter that searches only `spreadsheet` docs and ask: "What is the year-end inventory balance?"
2. Use `where={'source': '07_board_minutes.pdf'}` and ask about Q3 financial performance.


## Common errors

| Symptom | Fix |
|---|---|
| Filter returns 0 hits | Check the exact metadata value with `Counter` first (case matters). |
| Cross-doc answer misses one side | Increase `k` so both documents are likely to be retrieved. |


## ⚠️ Professional caution

When the system combines facts from two documents, *each* fact still needs verification against its source. A consistent-sounding combination can still contain a hallucinated link between unrelated facts.